In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import re
import pandas as pd
from glob import glob

import scienceplots
plt.style.use("science")

In [ ]:
SENSOR_DELAYS = {
    "camera": 0,
    "light": 0,
    "temperature": 5200,
    "mic": 200
}

In [ ]:
def parse_pcap(path):
    data = []

    fh = open(path)
    packets = json.load(fh)
    fh.close()

    start_time = None
    found_res = False
    finish_time = None

    for packet in packets:
        if start_time is not None and found_res and "_source" in packet and "layers" in packet["_source"] and "mqtt" in packet["_source"]["layers"] \
            and "mqtt.hdrflags_tree" in packet["_source"]["layers"]["mqtt"] and "mqtt.msgtype" in packet["_source"]["layers"]["mqtt"]["mqtt.hdrflags_tree"] \
            and packet["_source"]["layers"]["mqtt"]["mqtt.hdrflags_tree"]["mqtt.msgtype"] == "3":
                data.append({
                    "start": start_time,
                    "finish": finish_time
                })
                
                start_time = None
                found_res = False
                finish_time = None
        
        if start_time is not None and not found_res and "_source" in packet and "layers" in packet["_source"] and "mqtt" in packet["_source"]["layers"] \
            and "mqtt.hdrflags_tree" in packet["_source"]["layers"]["mqtt"] and "mqtt.msgtype" in packet["_source"]["layers"]["mqtt"]["mqtt.hdrflags_tree"] \
            and packet["_source"]["layers"]["mqtt"]["mqtt.hdrflags_tree"]["mqtt.msgtype"] == "3":
                found_res = True
                
        if start_time is None and not found_res and "_source" in packet and "layers" in packet["_source"] and "mqtt" in packet["_source"]["layers"] \
            and "mqtt.hdrflags_tree" in packet["_source"]["layers"]["mqtt"] and "mqtt.msgtype" in packet["_source"]["layers"]["mqtt"]["mqtt.hdrflags_tree"] \
            and packet["_source"]["layers"]["mqtt"]["mqtt.hdrflags_tree"]["mqtt.msgtype"] == "3":   
                start_time = pd.to_datetime(packet["_source"]["layers"]["frame"]["frame.time_epoch"], format = "%Y-%m-%dT%H:%M:%S.%fZ")
        
        # Frame and frame.time_epoch are usually in layers if it exist, so we skip checking them
        if found_res and "_source" in packet and "layers" in packet["_source"]:
            finish_time = pd.to_datetime(packet["_source"]["layers"]["frame"]["frame.time_epoch"], format = "%Y-%m-%dT%H:%M:%S.%fZ")
            
    data = data[1:]
    
    if "light" in path:
        data = data[1:]
    
    if "light_wall" in path:
        data = data[1:]

    df = pd.DataFrame(data)
    df["duration"] = (df["finish"] - df["start"]).dt.total_seconds() * 1000
    if "wall" in path:
        df["sensor"] = path.split("_")[0]
        df["wall"] = True
    else:
        df["sensor"] = path[:-5]
        df["wall"] = False
    
    df["delay"] = df["sensor"].map(lambda x: SENSOR_DELAYS[x])
    
    return df

In [ ]:
def parse_iot_log(path):
    fh = open(path)

    sensor_times = []
    encoding_times = []

    sensor_regex = re.compile(r"Time taken for .+: (\d+)")
    protobuf_regex = re.compile(r"Time taken for \w+ protobuf encoding: (\d+)")
    for line in fh.readlines():
        if "initialization" in line:
            continue
        
        proto_serialization_time = protobuf_regex.findall(line)
        if len(proto_serialization_time) != 0:
            encoding_times.append(int(proto_serialization_time[0]))
            continue
        
        sensor_time = sensor_regex.findall(line)
        if len(sensor_time) != 0:
            sensor_times.append(int(sensor_time[0]))
            
    return sensor_times[2:], encoding_times[2:]

In [ ]:
net_cap_dfs = []
for net_cap in glob("*.json"):
    df = parse_pcap(net_cap)
    net_cap_dfs.append(df)
    
df_net_cap : pd.DataFrame = pd.concat(net_cap_dfs).reset_index(drop = True)

In [ ]:
for iot_log_file in glob("*.txt"):
    wall = False
    sensor = iot_log_file[:-4]
    if "wall" in iot_log_file:
        wall = True
        sensor = iot_log_file.split("_")[0]
    
    sensor_times, encoding_times = parse_iot_log(iot_log_file)
    df_net_cap.loc[(df_net_cap["sensor"] == sensor) & (df_net_cap["wall"] == wall), "sensor_time"] = sensor_times
    df_net_cap.loc[(df_net_cap["sensor"] == sensor) & (df_net_cap["wall"] == wall), "encoding_time"] = encoding_times

df_net_cap["sensor_time"] = df_net_cap["sensor_time"] - df_net_cap["delay"]    
df_net_cap["other"] = df_net_cap["duration"] - df_net_cap["delay"] - df_net_cap["sensor_time"] - df_net_cap["encoding_time"]

In [ ]:
q1 = df_net_cap.groupby(["sensor", "wall"])["duration"].quantile(0.25)
q3 = df_net_cap.groupby(["sensor", "wall"])["duration"].quantile(0.75)
iqr = q3 - q1
lb = q1 - 1.5 * iqr
ub = q3 + 1.5 * iqr

df_bounds = df_net_cap.merge(lb.to_frame().rename({"duration": "lb"}, axis = 1), on = ["sensor", "wall"])
df_bounds = df_bounds.merge(ub.to_frame().rename({"duration": "ub"} , axis = 1), on = ["sensor", "wall"])

df_net_cap = df_bounds[(df_bounds["duration"] >= df_bounds["lb"]) & (df_bounds["duration"] <= df_bounds["ub"])].copy()
df_net_cap = df_net_cap[["start", "finish", "duration", "sensor", "wall", "delay", "sensor_time", "encoding_time", "other"]]

In [ ]:
df = df_net_cap.groupby(["sensor", "wall"])[["other", "delay", "sensor_time", "encoding_time"]].agg(["mean", "std"])
df.columns = [f"{i}_{j}" for i, j in df.columns]

df = df.reset_index()
df["total_std"] = df["other_std"] + df["delay_std"] + df["sensor_time_std"] + df["encoding_time_std"]
df = df.drop(["other_std", "delay_std", "sensor_time_std", "encoding_time_std"], axis = 1)

df["Setting"] = df["sensor"].str.capitalize() + " \n " + df["wall"].map({True: "With wall", False: "Without wall"})
df = df.drop(["sensor", "wall"], axis = 1)

In [ ]:
df_net_cap[df_net_cap["sensor"] == "mic"]["sensor_time"].mean()

In [ ]:
from scipy import stats
for idx, comb in df_net_cap.groupby(["sensor", "wall"]).size().reset_index().iterrows():
    sub_df = df_net_cap[(df_net_cap["sensor"] == comb["sensor"]) & (df_net_cap["wall"] == comb["wall"])]
    
    res_ks = stats.kstest(sub_df["duration"], stats.norm.cdf)
    print(res_ks)
    print(f"For {comb['sensor']} and wall {comb['wall']}, the data is {'NOT NORMAL' if res_ks.pvalue < 0.05 else 'NORMAL'}")
    
    plt.figure(figsize = (22, 8))
    sns.histplot(sub_df, x = "duration", kde=True)
    plt.title(f"Histogram for latency measurements for {comb['sensor'] if comb['sensor'] != 'mic' else 'microphone'} sensor {'with' if comb['wall'] else 'without'} a physical wall between Access Point and IoT Node")
    plt.xlabel("Duration (in milliseconds)")
    plt.savefig(f"histogram_{comb['sensor']}_{'with' if comb['wall'] else 'without'}_wall.pdf", format = "pdf")

In [ ]:
plt.rcParams.update({'font.size': 22})

In [ ]:
df = df.replace({"Mic \n Without wall": "Microphone \n Without wall", "Mic \n With wall": "Microphone \n With wall"})

In [ ]:
axes = df.plot(x = "Setting", kind = "bar", stacked = True, y = ["other_mean", "delay_mean", "sensor_time_mean", "encoding_time_mean"], figsize = (22, 8), 
        rot = 0, xlabel = "Sensor type and wall between Access Point and IoT Node", ylabel = "Duration (in milliseconds)", title = "Time required for end-to-end communication between Python Script - MQTT Broker - IoT Node")
for i, entry in enumerate(df.itertuples()):
    plt.errorbar(entry.Index, entry.other_mean + entry.delay_mean + entry.sensor_time_mean + entry.encoding_time_mean, entry.total_std, ecolor = "black")
    
plt.legend(["Other means (Transport Overhead, IoT Thread Scheduler, etc.)", "IoT - Hard-coded HAL_Delay Required For Sensors", "IoT - Sensor Reading Time", "IoT - Protocol Buffer Serialization Time"])

In [ ]:
axes.figure.savefig("benchmark_toolcall.pdf", format = "pdf")

In [ ]:
x = df_net_cap[(df_net_cap["sensor"] == "camera") & (df_net_cap["wall"] == False)]["duration"]
y = df_net_cap[(df_net_cap["sensor"] == "camera") & (df_net_cap["wall"] == True)]["duration"]
U, p = stats.mannwhitneyu(x, y)
print(f"For camera, the p-value is {p} with test statistic {U}, so the distributions are {'NOT' if p > 0.05 else ''} significant different")

x = df_net_cap[(df_net_cap["sensor"] == "light") & (df_net_cap["wall"] == False)]["duration"]
y = df_net_cap[(df_net_cap["sensor"] == "light") & (df_net_cap["wall"] == True)]["duration"]
U, p = stats.mannwhitneyu(x, y)
print(f"For light, the p-value is {p} with test statistic {U}, so the distributions are {'NOT' if p > 0.05 else ''} significant different")

x = df_net_cap[(df_net_cap["sensor"] == "mic") & (df_net_cap["wall"] == False)]["duration"]
y = df_net_cap[(df_net_cap["sensor"] == "mic") & (df_net_cap["wall"] == True)]["duration"]
U, p = stats.mannwhitneyu(x, y)
print(f"For mic, the p-value is {p} with test statistic {U}, so the distributions are {'NOT' if p > 0.05 else ''} significant different")

x = df_net_cap[(df_net_cap["sensor"] == "temperature") & (df_net_cap["wall"] == False)]["duration"]
y = df_net_cap[(df_net_cap["sensor"] == "temperature") & (df_net_cap["wall"] == True)]["duration"]
U, p = stats.mannwhitneyu(x, y)
print(f"For temperature, the p-value is {p} with test statistic {U}, so the distributions are {'NOT' if p > 0.05 else ''} significant different")